In [2]:
import yaml
from types import SimpleNamespace
from strategies.run_allen import get_allen_result, get_allen_signals
from utils.io import read_all_df
import pandas as pd
from evaluation.stats import sharpe

def load_config(path):
    with open(path, "r") as f:
        cfg_dict = yaml.safe_load(f)
    return SimpleNamespace(**cfg_dict)

backtest_config = load_config("config/backtest.yaml")

f = open("config/backtest.yaml")
cfg = yaml.safe_load(f)
result_dir = "results/Example_Result"

result = get_allen_result(cfg, result_dir)
signals = get_allen_signals(cfg, result_dir)

輸入成功!


In [3]:
test_pred, test_truth, _, _ = read_all_df(result_dir)

In [5]:
def yearly_returns(returns):
    yearly_roi = returns.resample("Y").last() / returns.resample("Y").first() - 1
    rois = {}
    for year in yearly_roi.index:
        rois[year.year] = yearly_roi[year]
    rois['final'] = returns.iloc[-1] / returns.iloc[0] - 1
    rois['sharpe'] = sharpe(returns)
    return rois

model_rois = yearly_returns(result['model'].returns)
baseline_rois = yearly_returns(result['benchmark'].returns)

print(model_rois, baseline_rois)

{2021: 0.7572661057660544, 2022: -0.10815061907293944, 2023: 0.7906979385577566, 2024: 0.12387087282961562, 'final': 2.1366930855910535, 'sharpe': 1.1646566686341118} {2021: 0.45978760395481344, 2022: -0.20070222104641622, 2023: 0.4638155086042668, 2024: 0.13995698204271623, 'final': 0.9508996418448139, 'sharpe': 0.9009005635855925}


In [6]:
from evaluation.surge_metrics import compute_surge_metrics_from_buy_signals

metric = compute_surge_metrics_from_buy_signals(
    test_pred=test_pred,
    test_truth=test_truth,
    buy_signals=signals['buy_signals'],
    threshold=0.3
)

model_data = {
    'category': 'example',
    'baseline': 0,
    'actual_surge_rate_on_buy_signals': metric['actual_surge_rate_on_buy_signals'],
}
model_data.update(model_rois)

baseline_data = {
    'category': 'example',
    'baseline': 1,
    'actual_surge_rate_on_buy_signals': metric['overall_actual_surge_rate'],
}
baseline_data.update(baseline_rois)

result_df = pd.DataFrame([model_data, baseline_data])
result_df


,category,baseline,actual_surge_rate_on_buy_signals,2021,2022,2023,2024,final,sharpe
0,example,0,0.142975,0.757266,-0.108151,0.790698,0.123871,2.136693,1.164657
1,example,1,0.069795,0.459788,-0.200702,0.463816,0.139957,0.950900,0.900901
